In [1]:
import pandas as pd
import glob
import os


# Path to your folder
raw_2022 = '../data/2022'
raw_2023 = '../data/2023'
raw_2024 = '../data/2024'
raw_2025 = '../data/2025'
# Get a list of all csv filenames
data_2022 = sorted(glob.glob(os.path.join(raw_2022, "*.csv")))
data_2023 = sorted(glob.glob(os.path.join(raw_2023, "*.csv")))
data_2024 = sorted(glob.glob(os.path.join(raw_2024, "*.csv")))
data_2025 = sorted(glob.glob(os.path.join(raw_2025, "*.csv")))
all_raw_data = [data_2022, data_2023, data_2024, data_2025]

In [2]:
# 1. Flatten your nested list into one single list of file paths
file_list = [file for year_list in all_raw_data for file in year_list]

# 2. Read all files into a list of DataFrames
# We use parse_dates here so the sorting works correctly later
temp_dfs = []
for file in file_list:
    df = pd.read_csv(file, parse_dates=['Date'], dayfirst=True)
    temp_dfs.append(df)

# 3. Stack them all on top of each other
final_merged_df = pd.concat(temp_dfs, ignore_index=True)

# Combine Date and Time into a single datetime column
final_merged_df['Date'] = pd.to_datetime(
    final_merged_df['Date'].astype(str).str.strip() + ' ' + final_merged_df['Time'].astype(str).str.strip(),
    errors='coerce'
)

# Floor the date to the nearest hour (important for hourly aggregation)
final_merged_df['Date'] = final_merged_df['Date'].dt.floor('h')

# Sort by Date (now includes Time)
final_merged_df = final_merged_df.sort_values(by='Date', ascending=True)

# 5. Clean up the index (so it goes 0, 1, 2... instead of jumping around)
final_merged_df = final_merged_df.reset_index(drop=True)

# Preview the sorted data
print(final_merged_df.head(20))

/tmp/ipykernel_151804/2741230692.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, parse_dates=['Date'], dayfirst=True)
/tmp/ipykernel_151804/2741230692.py:8: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, parse_dates=['Date'], dayfirst=True)


                  Date   Time           Order No Account No.  \
0  2022-01-01 09:00:00  09:05  12683966143262057        5804   
1  2022-01-01 09:00:00  09:05  12683966143262057        5804   
2  2022-01-01 09:00:00  09:05  12683966143262057        5804   
3  2022-01-01 09:00:00  09:39  12683966143262073        5805   
4  2022-01-01 09:00:00  09:39  12683966143262073        5805   
5  2022-01-01 09:00:00  09:39  12683966143262073        5805   
6  2022-01-01 09:00:00  09:40  12683966143262073        5805   
7  2022-01-01 09:00:00  09:40  12683966143262073        5805   
8  2022-01-01 10:00:00  10:53  12683966143262113        5807   
9  2022-01-01 10:00:00  10:54  12683966143262113        5807   
10 2022-01-01 10:00:00  10:53  12683966143262113        5807   
11 2022-01-01 10:00:00  10:53  12683966143262113        5807   
12 2022-01-01 10:00:00  10:41  12683966143262094        5806   
13 2022-01-01 10:00:00  10:41  12683966143262094        5806   
14 2022-01-01 10:00:00  10:41  126839661

In [3]:
final_merged_df

,Date,Time,Order No,Account No.,Account ID,Terminal,Type,Description,Destination,Detail,...,Discount,Service Charge,Tip,Payment Amount,Forfeit Amount,VAT,Other Tax,Cash Back,Change,Unnamed: 28
0,2022-01-01 09:00:00,09:05,12683966143262057,5804,12648781766153556,Till 1,Sale,Med Latte,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.30,0.0,0.0,0.0,NaN
1,2022-01-01 09:00:00,09:05,12683966143262057,5804,12648781766153556,Till 1,Sale,Sml Latte,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.27,0.0,0.0,0.0,NaN
2,2022-01-01 09:00:00,09:05,12683966143262057,5804,12648781766153556,Till 1,Payment,Close Accoun,NaN,21/9551,...,0.0,0.0,0.0,5.1,0.0,0.00,0.0,0.0,0.0,NaN
3,2022-01-01 09:00:00,09:39,12683966143262073,5805,12648781766153557,Till 1,Sale,Standard Tea,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.17,0.0,0.0,0.0,NaN
4,2022-01-01 09:00:00,09:39,12683966143262073,5805,12648781766153557,Till 1,Sale,F.Eggs on Toast,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.34,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1343083,2025-12-31 16:00:00,16:08,12683966149058749,6746,12648781766417416,Till 1,Payment,Close Accoun,NaN,21/1666,...,0.0,0.0,0.0,1.0,0.0,0.00,0.0,0.0,0.0,NaN
1343084,2025-12-31 16:00:00,16:12,12683966149058762,6747,12648781766417392,Till 1,Sale,Chicken Wrap,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.83,0.0,0.0,0.0,NaN
1343085,2025-12-31 16:00:00,16:12,12683966149058762,6747,12648781766417392,Till 1,Sale,Chips,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.46,0.0,0.0,0.0,NaN
1343086,2025-12-31 16:00:00,16:08,12683966149058724,6745,12648781766417415,Till 1,Sale,Ham Chse Toastie,Dine In,Standard,...,0.0,0.0,0.0,0.0,0.0,0.79,0.0,0.0,0.0,NaN


In [4]:
columns_to_keep = ['Date', 'Description', 'Quantity', 'Price']
final_merged_df = final_merged_df.filter(items=columns_to_keep)

In [5]:
final_merged_df

,Date,Description,Quantity
0,2022-01-01 09:00:00,Med Latte,1
1,2022-01-01 09:00:00,Sml Latte,1
2,2022-01-01 09:00:00,Close Accoun,0
3,2022-01-01 09:00:00,Standard Tea,1
4,2022-01-01 09:00:00,F.Eggs on Toast,1
...,...,...,...
1343083,2025-12-31 16:00:00,Close Accoun,0
1343084,2025-12-31 16:00:00,Chicken Wrap,1
1343085,2025-12-31 16:00:00,Chips,1
1343086,2025-12-31 16:00:00,Ham Chse Toastie,1


In [6]:
df_pivoted = final_merged_df.pivot_table(
    index='Date',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0  # This turns "NaN" (no sales) into 0
)

# 2. Clean up the formatting
# No need to reset_index here yet if we want to use .index.hour for filtering
# but we should remove the columns.name
df_pivoted.columns.name = None

In [7]:
df_pivoted.columns.to_list()

['10p Bag Charge',
 '25p Cup Discount',
 'Alm Flat White',
 'Almd & Aprct Bar',
 'Almon Chai Latte',
 'Almond',
 'Almond Iced Latt',
 'Almond Milk',
 'Almond Shot Pack',
 'Almond Tea',
 'Apl & Blk Smthie',
 'Apl & Blk Splash',
 'Apple',
 'Apple Juice',
 'Apple Warmer',
 'Armed Forces Day',
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 "B'currant Peeler",
 'BBQ Sauce',
 'BD Curry Roll',
 'BF IceMoc',
 'BLT',
 'Baby Mailer FSHD',
 'Babyccino',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bakewell Loaf',
 'Bakewell Mshake',
 'Banana Bread',
 'Banana Waln Cake',
 'Banoffee Shake',
 'Bberry Lemonade',
 'Bean Soldiers',
 'Beef Burger',
 'Beef Hors Foca',
 'Berry Fru Shoot',
 'Berry Raisn Shot',
 'Berry Tea',
 'Beyond Burger',
 'Big Breakfast',
 'Black',
 'Black Pudding',
 'Black Tea',
 'Blk Forest Bisc',
 'Blk Forest Syrup',
 'Blk Iced Amer',
 'Blueberry Muffin',
 'Bre Cran Toastie',
 'Bread Butter x2

In [8]:
columns_to_remove = [
     'Baby Mailer FSHD',
    '10p Bag Charge',
    '25p Cup Discount',
    'Black',
     'Chick Katsu Bag',
    'Cheeseboard Bag',
     'Close Accoun',
    'Clubcard Prices',
     'Cocoa Dusting',
     'Cream Tea Promo',
 'Fish Finger Bag',
    'Vegan Promo',
     'Mozz Pesto Bag',
     'Only Chck Burger',
     'Reusable Cup ',
    'JE Four Bundle',
    'JE Two bundle',
    'JE V\'day bundle',
    'Poppy Appeal',
    'Takeaway',
    'Lactofree',
     'Armed Forces Day',
    'with Brown Toast',
    'with Fried Bread',
    'with Lorne Saus',
    'with Mseed Toast',
    'with Poach Eggs',
    'with Poached Egg',
    'with Scramb Eggs',
    'with Scrambd Egg',
    'with Sdough +85p',
    'with White Toast'
]

# Identify which columns actually exist in the DataFrame to avoid KeyErrors
valid_initial_remove = [col for col in columns_to_remove if col in df_pivoted.columns]
df_pivoted.drop(columns=valid_initial_remove, inplace=True)

In [9]:
df_pivoted.shape

(13460, 773)

In [10]:
# Because of the sparsity of some products , had to be combined  into a single category
hot_drinks  = ['Babyccino','Berry Tea', 'Black Tea','Blk Iced Amer', 'Chai Latte', 'M Oat Pm HotC', 'M Skn YL HotC', 'M Soy Pm HotC', 'M Soy YL HotC', 'Sml LactF Capp', 'Soya Earl Grey', 'M Oat Pi Latt', 'M Skn Pi Latt', 'M Df Coc Vn Latt', 'M Df Hz Moch', 'M Oat Pr HotC', 'Soy Ceylon Tea', 'M Df Ml Moch', 'S Soy Ml Moch', 'Soya Babyccino', 'LacF Chai Latte', 'Med Lf Chai Latt', 'Sml Df LactF Lat', 'Lacf Earl Grey', 'S Pi Wh HotC', 'M Df Pn Latt', 'M Soy Pn Latt', 'Med Df Alm Latte', 'S Oat Ul HotC', 'M Coc Amer', 'M Coc Pr HotC', 'M Oat BF Latt', 'M Oat PC Latt', 'M Oat Pr Latt', 'S Df Oat Amer', 'M Df Oat PS Latt', 'Med St Gbr Latte', 'Double Macchiato', 'Lactofree Tea', 'Almond Tea',  'M Hc HotC', 'M Oat Vn Latt', 'Med Df LactF Lat', 'S Df Gb Latt', 'S Df Oat Gb Latt', 'S Hc HotC', 'Sml Almond Capp', 'Sml Almond HChoc', 'Sml Df Oat Mocha', 'Med LactoF Capp', 'S Ck HotC', 'Almon Chai Latte', 'M Coc CP Latt', 'Coc Decaf Tea', 'M Coc BF Latt', 'M Coc Gb Latt', 'Sml Soy Chai Lat', 'Df Coc FlatWhite', 'M SC Moch', 'Med Df Sk Mocha', 'Oat Chai Latte', 'Soya Decaf Tea', 'M Oat PS HotC', 'M Coc Chai Latt', 'M Coc Vn Latt', 'M Hz HotC', 'M Hz Latt', 'M Hz Moch', 'S Coc Amer', 'M Rasp HotC', 'Med Oat Df Mocha', 'M SC HotC', 'M SC Latt', 'S Coc Capp', 'S Df Ml Moch', 'S Ml HotC', 'S Ml Moch', 'S Soy PS Latt', 'M Ml HotC', 'M Ml Moch', 'Df Soy FlatWhite', 'Med LactoF Mocha', 'Single Macchiato', 'M CP Latt', 'M Coc HotC', 'M Df CP Latt', 'M Df Soy CP Latt', 'M Oat CP Latt', 'M Oat Pn Latt', 'M Skn CP Latt', 'M Skn Pn Latt', 'M Soy CP Latt', 'Med Df Soy Mocha', 'S Df Coc Capp', 'Skinny Earl Grey', 'Apple Warmer', 'S Ul HotC', 'Sml Alm Chai Lat', 'Sml Almond Mocha', 'Sml LactoF Latte', 'M Pi Wh HotC', 'M Ul HotC', 'Med Almond HChoc', 'M Soy PS Latt', 'M Soy ST Latt', 'Oat Earl Grey', 'S Df PS Latt', 'S Df ST Latt', 'S Df Skn PS Latt', 'S Oat PS Latt', 'S Oat ST HotC', 'S ST HotC', 'S ST Latt', 'Skinny Decaf Tea', 'Sml Sk Mocha', 'LactF Flat White', 'M Df Skn Amer', 'M ST HotC', 'M ST Latt','M Coc Capp', 'M Df Coc Capp', 'M Df Pr Latt', 'M PC Latt', 'M Pi Latt', 'M Pr HotC',  'M Pr Latt', 'Coc Tea', 'Small Alm Latte', 'Med Alm Mocha', 'S Gb Latt', 'S Oat Gb Latt', 'S Pm HotC', 'S YL HotC', 'Skinny', 'Std Assam Tea', 'Std Ceylon Tea', 'Std Darjee Tea', 'Std Rooibos Tea', 'Std Van Chai Tea', 'M Oat YL HotC', 'M Pm HotC', 'M YL HotC', 'Med Alm Chai Lat', 'Med LactoF Latte', 'Med Lactof HChoc', 'Decaf Tea', 'FineLemGin Tea', 'Cherry Warmer', 'Camomile Tea', 'Alm Flat White', 'S PS Latt', 'Sml Df Skin Capp', 'Oat Babyccino', 'M Df Gb Latt', 'M Df Oat Amer', 'M Gold HotC', 'M Oat BF HotC', 'M Skn Gb Latt', 'M Soy Amer', 'M Soy Gb Latt', 'Med Soya Mocha', 'Citrus Warmer', 'Sm Soya Hot Choc', 'Sml Df Soya Capp', 'Sml SK Chai Latt', 'Sngl Df Espresso', 'Oat Df Flat Whit', 'Oat Milk', 'Oat Tea', 'Med Alm Capp', 'Med Alm Latte', 'Med Df Alm Capp', 'Med Df Soya Capp', 'Med Oat Hot Choc', 'Med SK Chai Latt', 'Med Sk Hot Choc', 'Med Soya Chai La', 'M Ck HotC', 'Extra Decaf Shot', 'Standard Tea', 'Std Decaf Tea', 'Std Earl Grey', 'Soya Tea', 'Soya Flat White', 'Single Espresso', 'Skinny FlatWhite', 'Skinny Tea', 'Small Hot Choc', 'Small Oat Latte', 'Small Oat Mocha', 'Sml Americano', 'Sml Cappuccino', 'Sml Chai Latte', 'Sml Decaf Capp', 'Sml Df Americano', 'Sml Df Latte', 'Sml Df Mocha', 'Sml Df Oat Capp', 'Sml Df Oat Latte', 'Sml Df Sk Latte', 'Sml Df Soy Latte', 'Sml Latte', 'Sml Mocha', 'Sml Oat Capp', 'Sml Oat Chai Lat', 'Sml Oat Hot Choc', 'Sml Sk Capp', 'Sml Sk Hot Choc', 'Sml Skin Latte', 'Sml Soya Capp', 'Sml Soya Latte', 'Sml Soya Mocha', 'S Coc Latt', 'S Coc Moch', 'S Df Coc Latt', 'S Df Soy Amer', 'S Oat Amer', 'S Skn Amer', 'S Soy Amer', 'S Blk Amer', 'Orange Warmer', 'Oat Decaf Tea', 'Oat Flat White','Med Latte', 'Med Mocha', 'Med Oat Capp', 'Med Oat Chai Lat', 'Med Oat Latte', 'Med Oat Mocha', 'Med Skin Capp', 'Med Skinny Latte', 'Med Skinny Mocha', 'Med Soya Capp', 'Med Soya Latte', 'Medium Latte', 'Md Soya Hot Choc', 'Med Americano', 'Med Cappuccino', 'Med Cappucino', 'Med Chai Latte', 'Med Decaf Mocha', 'Med Df Americano', 'Med Df Capuccino', 'Med Df Latte', 'Med Df Oat Capp', 'Med Df Sk Latte', 'Med Df Skin Capp', 'Med Df Soy Latte', 'Med Hot Choc', 'M BF HotC', 'M BF Latt', 'M Blk Amer', 'M Coc Latt', 'M Coc Moch', 'M Df BF Latt', 'M Df Blk Amer', 'M Df Coc Latt', 'M Df Oat Latte', 'M Df PS Latt', 'M Df Soy Amer', 'M Gb Latt', 'M Hc Latt', 'M Oat Amer', 'M Oat Gb Latt', 'M Oat Hc Latt', 'M Oat PS Latt', 'M PS HotC', 'M PS Latt', 'M Pn Latt', 'M Skn Amer', 'M Skn BF Latt', 'Flat White', 'Fine Green Tea', 'FineJasmine Tea', 'FinePeppmint Tea', 'Extra Shot', 'Double Espresso', 'Dbl Df Espresso', 'Decaf Flat White', 'Df Sk Flat White',  'Coc FlatWhite']

soft_drinks = [ 'Berry Raisn Shot','Cawston E.Flower', 'Cawston Rhubarb','Cawston Apple','Cheese Sticks','Pink Lemon Soda', 'Coconut Water', 'Sparkling Water', 'Still Water', 'Sprite', 'San Pellegrino W', 'Oasis Citrus', 'Oasis Sum Fruit', 'Lem & Lime Water', 'Irn Bru', 'Fanta Orange', 'Fanta Zero', 'Diet Coke', 'Diet Irn Bru', 'Coke', 'Coke Zero', 'Cherry Kombucha',]

ice_drinks = ['BF IceMoc', 'Df Soya Ice Latt', 'Df Oat Iced Moch', 'Df Pi IceLat', 'Oat CocMa IceLat', 'Df Coco Ice Latt', 'Coc CocMa IceLat', 'Oat Pi IceLat', 'Skn Iced Flat', 'Mint Choc Mshake', 'Soy Iced Moch', 'Soy TM IceMatc', 'Soy Strb IceMatc', 'Coc Iced Moch', 'Df Iced Amer', 'Df Iced Flat', 'Df Soy Iced Flat', 'Df Soy Iced Amer', 'Hony Choc Mshake', 'Watermelon Crush', 'Straw Crm Frappe', 'LactoF Iced Latt', 'Strawb Frappe', 'Cloudy Lemonade', 'Hcomb Frappe', 'Oat TM IceMatc', 'Pi IceLat', 'Skn IceMatc', 'Coc TM IceMatc', 'Df Skn Iced Moch', 'Skn Iced Moch', 'Hzl Creme Mshake', 'Jammie Frappe', 'Pfruit Spritz', 'Choc Rasp Mshake', 'Df Sk Iced Latte', 'Salt Crml Mshake', 'Rhubarb Lemonade', 'Coc Iced Amer', 'Mill Milkshake', 'PS IceLat', 'Rhub Cust Frappe', 'Freaky Frappe', 'Oat CC IceMatc', 'Oat Iced Flat', 'Oat Iced Moch', 'Strawb Smthie', 'TM IceMatc', 'Skn Strb IceMatc', 'Soy IceMatc', 'Pineapple Smthie', 'Gingbread Shake', 'Df Coc Iced Latt', 'Df Iced Moch', 'CC IceMatc', 'Banoffee Shake', 'Ultim Choc Frapp', 'Pista Choc Frapp', 'Toffee Frappe', 'Strb IceMatc', 'S Coc Chai Latt', 'S Coc HotC', 'S Df Blk Amer', 'S Df Skn Amer', 'Sml Df Sk Mocha', 'Oat IceMatc', 'Oat Iced Amer', 'Oat Strb IceMatc', 'Peach Iced Tea', 'IceMatc', 'Iced Flat', 'Hazelnut Sbread', 'Df Iced Capp', 'Df Oat Iced Latt', 'CocMa IceLat', 'Coc IceMatc', 'Coc Strb IceMatc', 'Bberry Lemonade', 'Choc Pral Mshake', 'Bakewell Mshake', 'Watermelon Ice T', 'Cookies Frappe', 'Choc Shake', 'Yule Log Frappe', 'Almond Iced Latt', 'Gold Choc Frappe', 'Choc Hzl Mshake', 'Strawb Milkshake', 'Strawb Iced Tea', 'Skinny Ice Latte', 'Pecan & Pnut Bar', 'Pfruit Lime Cake', 'Elder Lemon Soda', 'Choc Milkshake', 'Cookie Frappe', 'Decaf Iced Latte', 'Dragon Cooler', 'Tof Apple Mshake', 'Soya Iced Latte', 'Oat Iced Latte', 'Med Iced Latte', 'Mango Smoothie', 'Honeycomb Mshake', 'Iced Amer', 'Iced Capp', 'Iced Latte', 'Iced Mango Crush', 'Iced Moch', 'Gbread Milkshake', 'Coc Iced Latt']

kids_shacks = [  'B\'currant Peeler','Apple','Bunny Cheese Bts','Cheese Puffs', 'Kids Raisins', 'Kids G\'bread Men', 'Strawb Jelly Pot', 'Strawb Peelers', 'Strawb Yoyo Bear', 'Rasp Jelly Pot', 'Orange', 'Kids Cheese Curl', 'Kids Breadsticks']

kids_drinks = ['Choc Milk Drink', 'Apple Juice','Berry Fru Shoot', 'Apl & Blk Splash','Water', 'Orange Juice', 'Orange Fru Shoot', 'Kids Orange Juic', 'Kids Apple Juice']

bakery = ['Blk Forest Bisc', 'Blueberry Muffin','Caramel Flapjack', 'Carrot Cake', 'Choc & Cranb Bar', 'Choc Teacake', 'Chocolate Muffin', 'Chocolate Tiffin', 'Chocolate Wafer','Finest Mince Pie', 'Choc Strawb Waff','Ice Cream HC Bun', 'Strawb Crm Cake', 'Rainbow Ice Loly', 'Carrot Mini Loaf', 'Flower Biscuit', 'Lemon Bakewell', 'Twirl Bites', 'Caramel Nibbles', 'Giant Buttons', 'Pineapple Pipstk', 'Pecan Pie', 'Jammie Biscuit', 'GF Choc Tart', 'Tiramisu Loaf', 'Jammie Blondie', 'Lem Rasp Biscuit', 'Love Heart Bisc', 'DE Choc Almonds', 'Cinnamon Bun', 'Bunny Biscuit', 'Choc Brioche', 'VicSponge Muffin', 'Bakewell Loaf', 'Lemon Choc Cake', 'KIND Caramel Bar', 'KIND Choc & Salt', 'GF Rasp Blondie', 'GF Vg Shortbread', 'Gingerbread Dino', 'Choc Shortbread', 'Chocolate Cookie', 'Choc Fudge Cake', 'Oaty Bar', 'Millionaire Cake', 'GF Brownie', 'Ginger Dunkers', 'Choc Crml Waffle', 'Choc Orange Bisc', 'Cashew Shot Pack', 'Toffee Mini Loaf','Gold Hcomb Tiff', 'Honeycomb Tiffin', 'Rocky Road', 'Shortbread', 'Mint Tiffin', 'Little Sundae', 'FruitNut Shot Pk', 'Eton Mess', 'Almond Shot Pack', 'Banana Waln Cake', 'Winter Bakewell', 'Vien Mince Pie', 'Ultimate Brownie', 'Tonys Choc Bar', 'Soreen Bar', 'Rudolph Biscuit', 'Shortbread Sq', 'Shortbrd Fingers', 'Rasp Shortbread', 'Protein Bar', 'Nut Pecan Shot', 'Mini Choc Cake', 'Lemon Muffin', 'Kit Kat', 'Gingerbread Bisc', 'Fruit Scone', 'GF Vg Mince Pie']
sauces = [
    'BBQ Sauce', 'Brown Sachet',
 'Brown Sauce', 'Mayo',
 'Mayo Sachet', 'Mustard Sachet', 'Tartar Sauce', 'Tomato Sachet',
 'Tomato Sauce',
]
syrup = [
 'Caramel Syrup', 'Hazelnut Syrup','S Caramel Syrup',
 'S.Caramel Syrup', 'SF Caramel Syrup',
 'SF Hazelnut Syrp',
 'SF Hazelnut Syru',
 'SF S Caramel Syr',
 'SF S.Caramel Syp',
 'SF Vanilla Syrup', 'Vanilla Syrup',
]
cheese = [
     'Cheese', 'Extra Cheese', 'Mature Cheese',
]
jam = [ 'Jam',
 'Jam Portion', 'Marmalade',]

milk = [
     'Almond',
 'Almond Milk',
 'Kids Milk',
     'Lactofree milk', 'Oat', 'Skinny Milk', 'Soya', 'Standard Milk',
 'Soya Milk',
]

# Identify which columns actually exist in the DataFrame to avoid KeyErrors
valid_hot_drinks = [col for col in hot_drinks if col in df_pivoted.columns]
valid_soft_drinks = [col for col in soft_drinks if col in df_pivoted.columns]
valid_ice_drinks = [col for col in ice_drinks if col in df_pivoted.columns]
valid_kids_shacks = [col for col in kids_shacks if col in df_pivoted.columns]
valid_kids_drinks = [col for col in kids_drinks if col in df_pivoted.columns]
valid_bakery = [col for col in bakery if col in df_pivoted.columns]
valid_sauces = [col for col in sauces if col in df_pivoted.columns]
valid_syrup = [col for col in syrup if col in df_pivoted.columns]
valid_cheese = [col for col in cheese if col in df_pivoted.columns]
valid_jam = [col for col in jam if col in df_pivoted.columns]
valid_milk = [col for col in milk if col in df_pivoted.columns]

df_pivoted['Hot Drinks'] = df_pivoted[valid_hot_drinks].sum(axis=1) if valid_hot_drinks else 0
df_pivoted['Soft Drinks'] = df_pivoted[valid_soft_drinks].sum(axis=1) if valid_soft_drinks else 0
df_pivoted['Ice Drinks'] = df_pivoted[valid_ice_drinks].sum(axis=1) if valid_ice_drinks else 0
df_pivoted['Kids Snacks'] = df_pivoted[valid_kids_shacks].sum(axis=1) if valid_kids_shacks else 0
df_pivoted['Kids Drinks'] = df_pivoted[valid_kids_drinks].sum(axis=1) if valid_kids_drinks else 0
df_pivoted['Bakery'] = df_pivoted[valid_bakery].sum(axis=1) if valid_bakery else 0
df_pivoted['Sauces'] = df_pivoted[valid_sauces].sum(axis=1) if valid_sauces else 0
df_pivoted['Syrups'] = df_pivoted[valid_syrup].sum(axis=1) if valid_syrup else 0
df_pivoted['Cheese Extra'] = df_pivoted[valid_cheese].sum(axis=1) if valid_cheese else 0
df_pivoted['Jams'] = df_pivoted[valid_jam].sum(axis=1) if valid_jam else 0
df_pivoted['Milks Extra'] = df_pivoted[valid_milk].sum(axis=1) if valid_milk else 0

columns_to_remove = (valid_hot_drinks + valid_soft_drinks + valid_ice_drinks + 
                     valid_kids_shacks + valid_kids_drinks + valid_bakery + 
                     valid_sauces + valid_syrup + valid_cheese + valid_jam + valid_milk)

df_pivoted.drop(columns_to_remove, axis=1, inplace=True)

In [11]:
df_pivoted

,Almd & Aprct Bar,Apl & Blk Smthie,Avo & Hal Muffin,"Avo, Egg & Bacon","Avo, Feta & Tom",Avocado on Toast,BD Curry Roll,BLT,Bacon,Bacon Bap,...,Soft Drinks,Ice Drinks,Kids Snacks,Kids Drinks,Bakery,Sauces,Syrups,Cheese Extra,Jams,Milks Extra
Date,,,,,,,,,,,,,,,,,,,,,
2022-01-01 09:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2022-01-01 10:00:00,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2022-01-01 11:00:00,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
2022-01-01 12:00:00,0,0,0,0,0,0,0,0,0,1,...,2,2,0,0,6,0,0,0,0,0
2022-01-01 13:00:00,0,0,0,0,0,0,0,0,0,1,...,5,0,2,4,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 12:00:00,0,0,0,1,0,0,1,0,0,1,...,18,2,0,7,15,0,4,1,0,0
2025-12-31 13:00:00,0,0,0,1,0,2,2,0,0,3,...,7,1,0,5,13,0,2,0,0,0
2025-12-31 14:00:00,0,0,0,2,0,0,1,0,2,4,...,3,1,2,5,5,0,1,0,0,0


In [12]:
white_toast = ['Bread Butter x2', 'White Toast' ,'Toast Butter x2', 'Toast x2' ,'Fried Bread',]
brown_toast = ['Brown Bread',  'Brown Toast']
porridge = ['Golden Porridge','GS Porridge Pot','Porridge Pot']
sourdough = ['Sourdough', 'Sourdough +85p']
multiseed = ['Multiseed Bread', 'Multiseed Toast']
mac = ['Spain Mac Cheese', 'Fest Mac Cheese','Mac & Cheese','Macaroni Cheese','Pigs Mac Cheese']
waffles = [ 'Caramel Waffle','Caramel Waffles',]

valid_white_toast = [col for col in white_toast if col in df_pivoted.columns]
valid_brown_toast = [col for col in brown_toast if col in df_pivoted.columns]
valid_porridge = [col for col in porridge if col in df_pivoted.columns]
valid_sourdough = [col for col in sourdough if col in df_pivoted.columns]
valid_multiseed = [col for col in multiseed if col in df_pivoted.columns]
valid_mac = [col for col in mac if col in df_pivoted.columns]
valid_waffles = [col for col in waffles if col in df_pivoted.columns]

df_pivoted['White Toast Bread'] = df_pivoted[valid_white_toast].sum(axis=1) if valid_white_toast else 0
df_pivoted['Brown Toast Bread'] = df_pivoted[valid_brown_toast].sum(axis=1) if valid_brown_toast else 0
df_pivoted['Porridge'] = df_pivoted[valid_porridge].sum(axis=1) if valid_porridge else 0
df_pivoted['Sourdough Toast Bread'] = df_pivoted[valid_sourdough].sum(axis=1) if valid_sourdough else 0
df_pivoted['Multiseed Toast Bread'] = df_pivoted[valid_multiseed].sum(axis=1) if valid_multiseed else 0
df_pivoted['Mac and Cheese'] = df_pivoted[valid_mac].sum(axis=1) if valid_mac else 0
df_pivoted['Waffles'] = df_pivoted[valid_waffles].sum(axis=1) if valid_waffles else 0

columns_to_remove = (valid_white_toast + valid_brown_toast + valid_porridge + 
                     valid_sourdough + valid_multiseed + valid_mac + valid_waffles)

df_pivoted.drop(columns_to_remove, axis=1, inplace=True)


In [13]:
# Filter for hours between 08:00 and 17:00
df_pivoted = df_pivoted[(df_pivoted.index.hour >= 8) & (df_pivoted.index.hour < 17)]

# Identify missing dates/hours
df_datetimes = df_pivoted.index

# Define the start and end dates/hours
start_dt = df_datetimes.min()
end_dt = df_datetimes.max()

# Generate the full range of hours
full_range = pd.date_range(start=start_dt.normalize(), end=end_dt.normalize() + pd.Timedelta(days=1), freq='h')
# Filter for 08:00 to 17:00 in the full range
full_range = full_range[(full_range.hour >= 8) & (full_range.hour < 17)]

# Identify missing datetimes
missing_datetimes = full_range.difference(df_datetimes)

print(f"Start Date: {start_dt.date()}")
print(f"End Date: {end_dt.date()}")
print(f"Total hours in range (08:00-17:00): {len(full_range)}")
print(f"Total hours present: {len(df_datetimes)}")
print(f"Number of missing datetimes: {len(missing_datetimes)}")

if len(missing_datetimes) > 0:
    # Create a DataFrame for missing datetimes with 0 for all item columns
    df_missing = pd.DataFrame(0, index=missing_datetimes, columns=df_pivoted.columns)

    # Combine with original data
    df_pivoted = pd.concat([df_pivoted, df_missing])

    # Sort again to maintain chronological order
    df_pivoted = df_pivoted.sort_index()

    print(f"Added {len(missing_datetimes)} missing hour rows with zero values.")
else:
    print("No missing hours to add.")

# Finally reset index so 'Date' (datetime) becomes a column
df_pivoted = df_pivoted.reset_index().rename(columns={'index': 'Date'})

# %%


Start Date: 2022-01-01
End Date: 2025-12-31
Total hours in range (08:00-17:00): 13149
Total hours present: 13001
Number of missing datetimes: 148
Added 148 missing hour rows with zero values.


In [14]:
df_pivoted

#df_pivoted.to_csv('processed_csv/sales_data_preprocessed.csv', index=False)

,Date,Almd & Aprct Bar,Apl & Blk Smthie,Avo & Hal Muffin,"Avo, Egg & Bacon","Avo, Feta & Tom",Avocado on Toast,BD Curry Roll,BLT,Bacon,...,Cheese Extra,Jams,Milks Extra,White Toast Bread,Brown Toast Bread,Porridge,Sourdough Toast Bread,Multiseed Toast Bread,Mac and Cheese,Waffles
0,2022-01-01 08:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2022-01-01 09:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,2022-01-01 10:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,2022-01-01 11:00:00,0,0,0,0,0,0,0,0,0,...,0,1,0,3,1,0,0,0,0,0
4,2022-01-01 12:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,2,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13144,2025-12-31 12:00:00,0,0,0,1,0,0,1,0,0,...,1,0,0,1,2,0,0,1,5,0
13145,2025-12-31 13:00:00,0,0,0,1,0,2,2,0,0,...,0,0,0,1,1,0,1,0,1,1
13146,2025-12-31 14:00:00,0,0,0,2,0,0,1,0,2,...,0,0,0,4,0,0,0,2,1,1
13147,2025-12-31 15:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0


In [15]:
df_pivoted.to_csv('processed_csv/sales_data_preprocessed.csv', index=False)


In [16]:
def check_missing_dates(df):
    """
    Checks if there are any missing hours (08:00-16:00) in the processed DataFrame.
    """
    if 'Date' not in df.columns:
        print("Error: 'Date' column not found in DataFrame.")
        return
    
    # Ensure Date is datetime
    df_dates = pd.to_datetime(df['Date'])
    
    start_dt = df_dates.min()
    end_dt = df_dates.max()
    
    if pd.isna(start_dt) or pd.isna(end_dt):
        print("Error: Could not determine start or end date.")
        return

    # Generate full range (08:00 to 16:00)
    full_range = pd.date_range(start=start_dt.normalize(), end=end_dt.normalize() + pd.Timedelta(days=1), freq='h')
    full_range = full_range[(full_range.hour >= 8) & (full_range.hour < 17)]
    
    missing = full_range.difference(df_dates)
    
    print(f"\n--- Final Data Check ---")
    print(f"Date Range: {start_dt} to {end_dt}")
    print(f"Expected Rows: {len(full_range)}")
    print(f"Actual Rows: {len(df_dates)}")
    
    if len(missing) > 0:
        print(f"WARNING: There are {len(missing)} missing timestamps!")
        if len(missing) > 10:
            print("First 10 missing:")
            print(missing[:10])
        else:
            print(missing)
    else:
        print("SUCCESS: No missing timestamps found in the 08:00-16:00 range.")

# Run the check
check_missing_dates(df_pivoted)



--- Final Data Check ---
Date Range: 2022-01-01 08:00:00 to 2025-12-31 16:00:00
Expected Rows: 13149
Actual Rows: 13149
SUCCESS: No missing timestamps found in the 08:00-16:00 range.
